In [2]:
from patient_rag import retrieve_patient_context

In [3]:
patient_id = "Adam631_Cronin387_aff8f143-2375-416f-901d-b0e4c73e3e58"

In [4]:
queries = [
    "What allergies do I have?",
    "Which vaccines did I receive in 2010?",
    "What was my blood pressure in 2010?"
]


for q in queries:
    print("\n==============================")
    print("Query:", q)
    
    result = retrieve_patient_context(
        patient_id=patient_id,
        query=q,
        top_k=3
    )
    
    print(result["context"])  



Query: What allergies do I have?
Encounter Date: 2000-01-02
Encounter Type: Encounter for symptom

Diagnoses:
- Perennial allergic rhinitis with seasonal variation (resolved)

Medications:
- Fexofenadine hydrochloride 30 MG Oral Tablet

Encounter Date: 2017-03-21
Encounter Type: Encounter for 'check-up'

Encounter Date: 2010-12-31
Encounter Type: Allergic disorder initial assessment

Vitals and Observations:
- Peanut IgE Ab in Serum: 0.15 kU/L
- Walnut IgE Ab in Serum: 0.1 kU/L
- Codfish IgE Ab in Serum: 0.17 kU/L
- Shrimp IgE Ab in Serum: 0.26 kU/L
- Wheat IgE Ab in Serum: 0.03 kU/L
- Egg white IgE Ab in Serum: 0.03 kU/L
- Soybean IgE Ab in Serum: 0.2 kU/L
- Cow milk IgE Ab in Serum: 0.25 kU/L
- White oak IgE Ab in Serum: 0.01 kU/L
- Common Ragweed IgE Ab in Serum: 0.01 kU/L
- Cat dander IgE Ab in Serum: 0.13 kU/L
- American house dust mite IgE Ab in Serum: 0.25 kU/L
- Cladosporium herbarum IgE Ab in Serum: 0.01 kU/L
- Honey bee IgE Ab in Serum: 0.02 kU/L
- Latex IgE Ab in Serum: 0.3

In [6]:
from patient_rag import load_patient_index, embed_query
import numpy as np
import faiss

index, documents = load_patient_index(patient_id)

query_vec = np.array([embed_query(q) for q in queries]).astype("float32")
faiss.normalize_L2(query_vec)

scores, indices = index.search(query_vec, 3)

print("Scores:", scores)
print("Indices:", indices)

Scores: [[0.6637825  0.55670905 0.5133534 ]
 [0.60450923 0.58211035 0.5812249 ]
 [0.53759646 0.517236   0.50433457]]
Indices: [[ 0 73  3]
 [ 2 23  0]
 [ 2  0  4]]


In [7]:
from unittest.mock import patch, MagicMock
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from rag_pipeline.patient_rag import retrieve_patient_context
import rag_pipeline.patient_rag as rag_module

# Đếm số lần embed_query được gọi
call_log = []
original_embed = rag_module.embed_query

def counting_embed(text):
    call_log.append(text[:40])  # log 40 chars đầu
    return original_embed(text)

# Test với query có year filter (worst case trước đây)
with patch.object(rag_module, "embed_query", side_effect=counting_embed):
    result = retrieve_patient_context(
        patient_id,
        query="What was my blood pressure in 2010?"
    )

print(f"✅ embed_query called {len(call_log)} time(s)")
print(f"   Calls: {call_log}")
print(f"\n📄 Retrieved {len(result['sources'])} documents")


✅ embed_query called 1 time(s)
   Calls: ['What was my blood pressure in 2010?']

📄 Retrieved 2 documents
